# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [ ]:
import pandas as pd
import re
from pandas.api.types import is_numeric_dtype, is_string_dtype

In [ ]:
# Load datasets
df_low = pd.read_csv('../data/raw/low_popularity_spotify_data.csv', low_memory=False)
df_high = pd.read_csv('../data/raw/high_popularity_spotify_data.csv', low_memory=False)
df_apr = pd.read_csv('../data/raw/SpotifyAudioFeaturesApril2019.csv', low_memory=False)
df_nov = pd.read_csv('../data/raw/SpotifyAudioFeaturesNov2018.csv', low_memory=False)

all_datasets = {'Low Popularity Songs': df_low, 
                'High Popularity Songs': df_high, 
                'Spotify Features April 2019': df_apr, 
                'Spotify Features Nov 2018': df_nov}

print("Dataset Shapes:")
for df_title, df in all_datasets.items():
    print(f"{df_title}: {df.shape}")

In [ ]:
# Converting the column names to lowercase
for df_title, df in all_datasets.items():
    df.columns = df.columns.str.strip().str.lower()
    print(f"{df_title}: {df.columns}")

In [ ]:
# Dropping unnamed columns
for df_title, df in all_datasets.items():
    print(df_title + " Dataset")
    print(f"Column count before dropping unnamed columns: {len(df.columns)}")
    df.drop(columns=df.columns[df.columns.str.contains("^unnamed")], inplace=True)
    print(f"Column count after dropping unnamed columns: {len(df.columns)}\n")

In [ ]:
# Finding the columns with object types
for df_title, df in all_datasets.items():
    object_cols = df.select_dtypes(include='object').columns
    print(f"Columns in {df_title} Dataset:\n{object_cols}\n")

In [ ]:
common_cols = list(set(df_apr.columns) & set(df_high.columns) & set(df_low.columns) & set(df_nov.columns))
print(f"Common columns across all datasets:\n{common_cols}")

Since we'll be combining these datasets later on, we'll focus on the object-type columns `track_id`, `artist_name` and `track_name`. We'll index each dataset using `track_id` since it's a common column across all datasets and likely to be unique as all of these datasets were created using Spotify data. We'll retain `artist_name` and `track_name` as they contain valuable information for the user once our model outputs a playlist. 

In [ ]:
# Updating types for the `artist_name` and `track_name` columns to string
for df_title, df in all_datasets.items():
    if "track_name" in df.columns:
        df["track_name"] = df["track_name"].astype(str).str.strip().str.lower()
    if "artist_name" in df.columns:
        df["artist_name"] = df["artist_name"].astype(str).str.strip().str.lower()

In [ ]:
# Adding a column to each dataset to reference its original source
df_low["source"] = "low_popularity"
df_high["source"] = "high_popularity"
df_apr["source"] = "april_2019"
df_nov["source"] = "nov_2018"

In [ ]:
# Concatenating all the datasets
combined_df = pd.concat(
    [df_high, df_low, df_nov, df_apr], 
    axis=0, 
    ignore_index=True, 
    sort=False
)

# Ensuring that the datasets are groupyed by their `track_id``
combined_df = combined_df.groupby('track_id', as_index=False).first()

# Viewing the resulting dimensions of the dataset
print(combined_df.shape)

In [ ]:
# Checking the distribution of the data's sources
print(combined_df["source"].value_counts())

In [ ]:
# Viewing how the combined dataset looks
combined_df.head()

In [ ]:
combined_df.info()

In [ ]:
combined_df.isnull().sum()

In [ ]:
# Dropping columns if more than 60% of the column consists of null values
print(f"Number of columns before drop: {len(combined_df.columns)}")

threshold = 0.6

combined_df = combined_df.drop(columns=[
        col 
        for col in combined_df.columns 
        if combined_df[col].isnull().sum() / len(combined_df) > threshold
    ]
)

print(f"Number of columns after drop: {len(combined_df.columns)}")

In [ ]:
# Checking that little to no null values remain
combined_df.isnull().sum()

In [ ]:
# Checking the column types
combined_df.info()

In [ ]:
# Correcting the object type columns with string types
combined_df = combined_df.astype({
    'track_id': 'string',
    'track_name': 'string',
    'source': 'string',
    'artist_name': 'string',
})

The columns were correctly converted to the proper types.

In [ ]:
combined_df.info()

In [ ]:
# Checking the statistical properties of the unscaled data
combined_df.describe()

In [ ]:
# Min-Max scaling each column to ensure that column values remain between 0 and 1, reducing feature dominance
for col in combined_df.columns:
    if is_numeric_dtype(combined_df[col]):
        min_val = combined_df[col].min()
        max_val = combined_df[col].max()
        combined_df[col] = (combined_df[col] - min_val) / (max_val - min_val)

In [ ]:
# Verifying that columns were Min-Max scaled
combined_df.describe()

Each column has a minimum and maximum of 0 and 1 respectively, verifying that Min-Max scaling was properly performed.

In [ ]:
# Imputing data where data is null
for col in combined_df.columns:
    if is_numeric_dtype(combined_df[col]):
        # Fill null numeric columns with the column median
        combined_df[col] = combined_df[col].fillna(combined_df[col].median())
    elif is_string_dtype(combined_df[col]):
        # Fill null string type columns with an empty string
        combined_df[col] = combined_df[col].fillna("")

In [ ]:
combined_df.isnull().sum()

All null values were taken care of.

In [ ]:
# Dropping duplicate rows
print(f"Before dropping duplicates: {combined_df.shape}")

# Dropping rows with the same track name and artist name
combined_df = combined_df.drop_duplicates(subset=['track_name', 'artist_name']) 

print(f"After dropping duplicates: {combined_df.shape}")

In [ ]:
# Dropping track id and grouping by `track_name` since we know it's unique now
combined_df = combined_df.groupby('track_name', as_index=False).first()
combined_df = combined_df.drop(columns=['track_id'])

In [ ]:
# Verifying that `track_id` was dropped and that the dataset is grouped by `track_name`
combined_df.head()

In [ ]:
# Removing punctuation and making cleaned text columns
def keep_alphanum(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

combined_df['track_name_clean'] = combined_df['track_name'].apply(keep_alphanum)
combined_df['artist_name_clean'] = combined_df['artist_name'].apply(keep_alphanum)

In [ ]:
combined_df.head()

In [ ]:
# Writing processed dataset to a CSV file
combined_df.to_csv("../data/cleaned/spotify_master_clean.csv", index=False)